# Análise de Regressão dos Dados — CTI Global
**Projeto Interdisciplinar – 4º CCOMP – Análise Inferencial de Dados – Entrega 02**

**Pergunta central:** o lucro da CTI vira caixa? Ou seja, o EBITDA explica a Geração de Caixa nos 1.200 cenários da simulação de Monte Carlo?

**O que este notebook faz**
1. Carrega o CSV bruto e monta a base (1 linha por cenário e ano).
2. Verifica como a Geração de Caixa é calculada (identidade contábil) e quais variáveis realmente variam entre os cenários.
3. Ajusta modelos de regressão linear (OLS): simples, múltiplo por ano, múltiplo com todos os anos 1 a 11, e um modelo separado para o Ano 12.
4. Interpreta coeficientes, R², p-valores e intervalos de confiança.
5. Faz diagnóstico dos resíduos, multicolinearidade e validação em cenários que o modelo não viu.
6. Registra as limitações.


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
os.makedirs('figuras', exist_ok=True)
os.makedirs('tabelas', exist_ok=True)

AZUL, LARANJA, CINZA, VERDE = '#1F3864', '#C55A11', '#7F7F7F', '#548235'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titlesize': 12, 'axes.labelsize': 10, 'font.size': 10})
pd.options.display.float_format = '{:,.3f}'.format
pd.options.display.width = 200
pd.options.display.max_columns = 40

## 1. Carga dos dados

In [ ]:
ARQUIVO = 'Demonstrativo_Fecap_v3.csv'

if not os.path.exists(ARQUIVO):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Coloque o arquivo Demonstrativo_Fecap_v3.csv na mesma pasta do notebook.')

longo = pd.read_csv(ARQUIVO, sep=';', encoding='latin-1', header=None,
                    names=['ano', 'cen', 'conta', 'valor'], dtype=str)

longo = longo[longo['conta'].notna()].copy()
longo['valor'] = (longo['valor'].str.replace('.', '', regex=False)
                                .str.replace(',', '.', regex=False).astype(float))

print('Linhas:', len(longo), '| Cenários:', longo['cen'].nunique(),
      '| Anos:', longo['ano'].nunique(), '| Contas:', longo['conta'].nunique())
print('Linhas duplicadas (ano, cenário, conta):', longo.duplicated(['ano', 'cen', 'conta']).sum())

In [ ]:
largo = longo.pivot_table(index=['cen', 'ano'], columns='conta', values='valor', aggfunc='first').reset_index()
largo['n'] = largo['ano'].str.extract(r'(\d+)')[0].astype(int)
print(largo.shape)

## 2. Variáveis do modelo
- **G (Y):** `FLU - Geração de Caixa`
- **EBITDA:** Resultado Operacional + |Depreciação e Amortização| (recalculado)
- **CAPEX:** |`FLU - Investimentos`|
- **DESPFIN:** |`FLU - Despesas Financeiras`|
- **DIST:** |`FLU - Distribuição para Acionista`|
- **IR:** |`FLU - Imposto de Renda e Contribuição Social`| (usado só para conferência, explicado abaixo)

As saídas de caixa entram em valor absoluto. Assim, um coeficiente negativo quer dizer "quanto maior a saída, menor o caixa".

In [ ]:
M = 1e6
base = pd.DataFrame({
    'cen': largo['cen'],
    'n': largo['n'],
    'G': largo['FLU - Geração de Caixa'] / M,
    'EBITDA': (largo['DRE - Resultado Operacional'] + largo['DRE - Depreciação e Amortização'].abs()) / M,
    'CAPEX': largo['FLU - Investimentos'].abs() / M,
    'DESPFIN': largo['FLU - Despesas Financeiras'].abs() / M,
    'DIST': largo['FLU - Distribuição para Acionista'].abs() / M,
    'IR': largo['FLU - Imposto de Renda e Contribuição Social'].abs() / M,
})
base.to_csv('tabelas/base_regressao.csv', index=False)
base.groupby('n')[['G', 'EBITDA', 'CAPEX', 'DESPFIN', 'DIST']].mean().round(1)

### 2.1 A Geração de Caixa é uma soma de itens do fluxo de caixa
Antes de modelar, vale testar se a Geração de Caixa é só a soma dos itens do fluxo. Se for, uma regressão com todos esses itens teria R² = 1 por pura contabilidade, e isso não ensinaria nada.

In [ ]:
comp = ['FLU - Receita', 'FLU - Tributos', 'FLU - Custos', 'FLU - Investimentos', 'FLU - Entradas',
        'FLU - Despesas Financeiras', 'FLU - Imposto de Renda e Contribuição Social',
        'FLU - Distribuição para Acionista']

dif = (largo[comp].sum(axis=1) - largo['FLU - Geração de Caixa']).abs()
print('Maior diferença entre a soma dos itens e a Geração de Caixa (R$):', round(dif.max(), 6))

m_id = sm.OLS(largo['FLU - Geração de Caixa'] / M, sm.add_constant(largo[comp] / M)).fit()
print('R² da regressão com todos os itens do fluxo:', round(m_id.rsquared, 6))

**Conclusão:** a Geração de Caixa é exatamente a soma dos itens do fluxo. Por isso o EBITDA (que está na DRE, não no fluxo) é a variável interessante para a pergunta "o lucro vira caixa?". CAPEX e despesas financeiras entram como controles, porque são itens de caixa que a DRE não mostra do mesmo jeito.
O Imposto de Renda é derivado do lucro e tem correlação forte com o EBITDA, então ficou fora para não duplicar informação.

### 2.2 O que realmente varia entre cenários dentro de cada ano?
Em uma regressão, uma variável só ajuda a explicar a diferença entre cenários se ela **muda de um cenário para o outro**. Abaixo, o coeficiente de variação (desvio-padrão ÷ média, em %) dentro de cada ano.

In [ ]:
vars_x = ['EBITDA', 'CAPEX', 'DESPFIN', 'DIST']
cv = (base.groupby('n')[vars_x].std() / base.groupby('n')[vars_x].mean().abs() * 100).round(2)
cv.index.name = 'Ano'
cv.to_csv('tabelas/tabela_cv_por_ano.csv')
cv

**Achado importante:** a Distribuição para Acionista (`DIST`) tem variação **zero** em quase todos os anos, ou seja, é uma política fixa (igual em todos os 1.200 cenários). Ela só varia nos Anos 4 e 12. Por isso:
- nos anos em que `DIST` é constante, ela **não entra** no modelo (seria colinear com o intercepto);
- o EBITDA varia pouco entre cenários (cerca de 0,6% a 3%), enquanto o CAPEX varia bastante (cerca de 9% a 26%).

## 3. Modelo 1 — regressão simples: Geração de Caixa ~ EBITDA
É o teste direto da tese: se o lucro virasse caixa, o EBITDA deveria explicar boa parte da Geração de Caixa.
Todos os modelos usam erros-padrão robustos (HC3), porque os resíduos não têm variância constante (confirmado no diagnóstico).

In [ ]:
def regressao_por_ano(formula, variaveis, anos):
    linhas = []
    for n in anos:
        d = base[base['n'] == n]
        m = smf.ols(formula, d).fit(cov_type='HC3')
        ci = m.conf_int()
        linha = {'Ano': n, 'N': int(m.nobs), 'R2': m.rsquared, 'R2_aj': m.rsquared_adj,
                 'RMSE': np.sqrt(m.ssr / m.nobs)}
        for v in variaveis:
            linha['coef_' + v] = m.params[v]
            linha['p_' + v] = m.pvalues[v]
            linha['icinf_' + v] = ci.loc[v, 0]
            linha['icsup_' + v] = ci.loc[v, 1]
        linhas.append(linha)
    return pd.DataFrame(linhas)

anos_ok = list(range(1, 12))

b11 = base[base['n'] <= 11].copy()
m1_pool = smf.ols('G ~ EBITDA', b11).fit(cov_type='HC3')
print('Modelo 1 com os anos 1 a 11 juntos (sem separar por ano)')
print('  coeficiente do EBITDA:', round(m1_pool.params['EBITDA'], 3), '| p-valor:', m1_pool.pvalues['EBITDA'])
print('  R²:', round(m1_pool.rsquared, 3), '| N:', int(m1_pool.nobs))

t1 = regressao_por_ano('G ~ EBITDA', ['EBITDA'], anos_ok)
t1.to_csv('tabelas/tabela_modelo1_por_ano.csv', index=False)
t1[['Ano', 'N', 'R2', 'R2_aj', 'RMSE', 'coef_EBITDA', 'p_EBITDA']]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

amostra = b11.sample(4000, random_state=1)
sc = ax[0].scatter(amostra['EBITDA'], amostra['G'], c=amostra['n'], cmap='viridis', s=8, alpha=0.6)
x = np.linspace(b11['EBITDA'].min(), b11['EBITDA'].max(), 50)
ax[0].plot(x, m1_pool.params['Intercept'] + m1_pool.params['EBITDA'] * x, color=LARANJA, lw=2)
ax[0].set_xlabel('EBITDA (R$ milhões)')
ax[0].set_ylabel('Geração de Caixa (R$ milhões)')
ax[0].set_title('Anos 1 a 11 juntos (R² = %.2f)' % m1_pool.rsquared)
plt.colorbar(sc, ax=ax[0], label='Ano')

d8 = base[base['n'] == 8]
ax[1].scatter(d8['EBITDA'], d8['G'], s=8, alpha=0.6, color=AZUL)
m8 = smf.ols('G ~ EBITDA', d8).fit()
xx = np.linspace(d8['EBITDA'].min(), d8['EBITDA'].max(), 50)
ax[1].plot(xx, m8.params['Intercept'] + m8.params['EBITDA'] * xx, color=LARANJA, lw=2)
ax[1].set_xlabel('EBITDA (R$ milhões)')
ax[1].set_ylabel('Geração de Caixa (R$ milhões)')
ax[1].set_title('Somente o Ano 8 (R² = %.2f)' % m8.rsquared)

plt.tight_layout()
plt.savefig('figuras/fig01_dispersao_ebitda_caixa.png', dpi=150)
plt.show()

## 4. Modelo 2 — regressão múltipla por ano: Geração de Caixa ~ EBITDA + CAPEX + DESPFIN
Aqui entram os dois itens de caixa que variam entre cenários. Ajustamos um modelo para cada ano de 1 a 11, comparando cenários do mesmo ano (assim o crescimento natural do EBITDA ao longo do tempo não mistura os anos).

In [ ]:
t2 = regressao_por_ano('G ~ EBITDA + CAPEX + DESPFIN', ['EBITDA', 'CAPEX', 'DESPFIN'], anos_ok)
t2.to_csv('tabelas/tabela_modelo2_por_ano.csv', index=False)
t2[['Ano', 'R2', 'R2_aj', 'RMSE', 'coef_EBITDA', 'p_EBITDA', 'coef_CAPEX', 'p_CAPEX', 'coef_DESPFIN', 'p_DESPFIN']]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
larg = 0.38
ax.bar(t1['Ano'] - larg / 2, t1['R2'], larg, color=CINZA, label='Modelo 1: só EBITDA')
ax.bar(t2['Ano'] + larg / 2, t2['R2'], larg, color=AZUL, label='Modelo 2: EBITDA + CAPEX + Desp. financeiras')
ax.set_xticks(anos_ok)
ax.set_xlabel('Ano')
ax.set_ylabel('R²')
ax.set_ylim(0, 1.05)
ax.set_title('Quanto da variação da Geração de Caixa entre cenários cada modelo explica')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('figuras/fig02_r2_por_ano.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, v, cor, tit in [(ax[0], 'EBITDA', AZUL, 'Coeficiente do EBITDA'), (ax[1], 'CAPEX', LARANJA, 'Coeficiente do CAPEX')]:
    err = [t2['coef_' + v] - t2['icinf_' + v], t2['icsup_' + v] - t2['coef_' + v]]
    a.errorbar(t2['Ano'], t2['coef_' + v], yerr=err, fmt='o', color=cor, capsize=3)
    a.axhline(0, color='black', lw=0.8)
    a.set_xticks(anos_ok)
    a.set_xlabel('Ano')
    a.set_ylabel('Caixa a mais por R\$ 1 a mais (R\$)')
    a.set_title(tit + ' por ano (IC 95%)')
plt.tight_layout()
plt.savefig('figuras/fig03_coeficientes_por_ano.png', dpi=150)
plt.show()

**Como ler:** o coeficiente do EBITDA diz quantos reais de caixa a mais aparecem, em média, para cada R$ 1 a mais de EBITDA, **mantendo CAPEX e despesas financeiras fixos**. Se o lucro virasse caixa 1 para 1, o coeficiente seria perto de 1. Na maioria dos anos ele fica entre 0,55 e 0,82, e nos Anos 4 e 5 ele fica **negativo**: nesses anos os cenários com mais EBITDA têm menos caixa, depois de descontar CAPEX e despesas financeiras.
Os coeficientes do CAPEX mudam de tamanho e até de sinal entre os anos. Isso mostra que, dentro de cada ano, o CAPEX também carrega o efeito de outros fatores do cenário (por exemplo, o calendário da dívida), e por isso ele não deve ser lido como causa.

## 5. Modelo 3 — todos os anos 1 a 11 juntos, com efeito fixo de ano
Aqui os 11 anos entram no mesmo modelo, com uma variável indicadora para cada ano (`C(n)`). Isso tira o efeito "ano" e deixa o modelo olhar para as diferenças entre cenários. Como o mesmo cenário aparece em vários anos, os erros-padrão são agrupados por cenário.

O R² total desse modelo é inflado pelos anos (o caixa médio muda muito de um ano para outro). Por isso também calculamos o **R² interno** (within), que mede só o que o modelo explica **além** das médias anuais.

In [ ]:
b11['cen_id'] = pd.factorize(b11['cen'])[0]
formula3 = 'G ~ EBITDA + CAPEX + DESPFIN + C(n)'

m3_ols = smf.ols(formula3, b11).fit()
m3 = smf.ols(formula3, b11).fit(cov_type='cluster', cov_kwds={'groups': b11['cen_id']})
m0 = smf.ols('G ~ C(n)', b11).fit()

r2_within = 1 - m3_ols.ssr / m0.ssr
print('R² total do Modelo 3:', round(m3_ols.rsquared, 4))
print('R² só com o efeito de ano:', round(m0.rsquared, 4))
print('R² interno (within):', round(r2_within, 4))
print('RMSE:', round(np.sqrt(m3_ols.ssr / m3_ols.nobs), 2), 'R$ milhões')

dm = b11[['EBITDA', 'CAPEX', 'DESPFIN']] - b11.groupby('n')[['EBITDA', 'CAPEX', 'DESPFIN']].transform('mean')
ci3 = m3.conf_int()
t3 = pd.DataFrame({
    'variavel': ['EBITDA', 'CAPEX', 'DESPFIN'],
    'coef': [m3.params[v] for v in ['EBITDA', 'CAPEX', 'DESPFIN']],
    'erro_padrao': [m3.bse[v] for v in ['EBITDA', 'CAPEX', 'DESPFIN']],
    'p_valor': [m3.pvalues[v] for v in ['EBITDA', 'CAPEX', 'DESPFIN']],
    'ic_inf': [ci3.loc[v, 0] for v in ['EBITDA', 'CAPEX', 'DESPFIN']],
    'ic_sup': [ci3.loc[v, 1] for v in ['EBITDA', 'CAPEX', 'DESPFIN']],
    'efeito_1dp': [m3.params[v] * dm[v].std() for v in ['EBITDA', 'CAPEX', 'DESPFIN']],
})
t3['R2_total'] = m3_ols.rsquared
t3['R2_ano_apenas'] = m0.rsquared
t3['R2_within'] = r2_within
t3.to_csv('tabelas/tabela_modelo3.csv', index=False)
t3

A coluna `efeito_1dp` mostra o quanto o caixa muda (em R$ milhões) quando a variável sobe um desvio-padrão **dentro do ano**. Ela serve para comparar a força de cada variável, já que EBITDA e CAPEX estão em escalas diferentes.

## 6. Ano 12 (fim da concessão) — modelo separado
No Ano 12 há liquidação de direitos e obrigações e 27 contas do balanço não existem (N/D). Por isso ele não entra no Modelo 3. Aqui a `DIST` varia entre cenários, então ela pode ser testada.

In [ ]:
def ajustar_ano(n, formula, nome):
    d = base[base['n'] == n]
    m = smf.ols(formula, d).fit(cov_type='HC3')
    linha = {'Ano': n, 'Modelo': nome, 'R2': m.rsquared, 'R2_aj': m.rsquared_adj, 'RMSE': np.sqrt(m.ssr / m.nobs)}
    for v in ['EBITDA', 'DIST', 'CAPEX', 'DESPFIN']:
        linha['coef_' + v] = m.params.get(v, np.nan)
        linha['p_' + v] = m.pvalues.get(v, np.nan)
    return linha

specs = [('A: só EBITDA', 'G ~ EBITDA'),
         ('B: só DIST', 'G ~ DIST'),
         ('C: EBITDA + DIST', 'G ~ EBITDA + DIST'),
         ('D: EBITDA + DIST + CAPEX', 'G ~ EBITDA + DIST + CAPEX'),
         ('E: EBITDA + DIST + CAPEX + DESPFIN', 'G ~ EBITDA + DIST + CAPEX + DESPFIN')]
t12 = pd.DataFrame([ajustar_ano(12, f, nome) for nome, f in specs])
t4 = pd.DataFrame([ajustar_ano(4, 'G ~ EBITDA + DIST + CAPEX + DESPFIN', 'Ano 4: EBITDA + DIST + CAPEX + DESPFIN')])
pd.concat([t12, t4]).to_csv('tabelas/tabela_ano12.csv', index=False)
pd.concat([t12, t4])

In [ ]:
d12 = base[base['n'] == 12]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, v, cor in [(ax[0], 'EBITDA', AZUL), (ax[1], 'DIST', LARANJA)]:
    a.scatter(d12[v], d12['G'], s=8, alpha=0.6, color=cor)
    mm = smf.ols('G ~ ' + v, d12).fit()
    xx = np.linspace(d12[v].min(), d12[v].max(), 50)
    a.plot(xx, mm.params['Intercept'] + mm.params[v] * xx, color='black', lw=1.5)
    a.set_xlabel(v + ' (R$ milhões)')
    a.set_ylabel('Geração de Caixa (R$ milhões)')
    a.set_title('Ano 12: Caixa x %s (R² = %.2f)' % (v, mm.rsquared))
plt.tight_layout()
plt.savefig('figuras/fig04_ano12.png', dpi=150)
plt.show()

## 7. Diagnóstico do Modelo 3
Verificamos os pressupostos da regressão linear: multicolinearidade (VIF), variância constante dos resíduos (Breusch-Pagan), normalidade (Jarque-Bera, assimetria, curtose), pontos influentes (distância de Cook) e dependência entre anos do mesmo cenário.

In [ ]:
Xv = sm.add_constant(dm)
vif = pd.DataFrame({'variavel': ['EBITDA', 'CAPEX', 'DESPFIN'],
                    'VIF': [variance_inflation_factor(Xv.values, i) for i in range(1, 4)]})
vif.to_csv('tabelas/tabela_vif.csv', index=False)

res = m3_ols.resid
bp = het_breuschpagan(res, m3_ols.model.exog)
jb = stats.jarque_bera(res)
cook = m3_ols.get_influence().cooks_distance[0]
limite_cook = 4 / len(b11)

rp = b11.assign(res=res).pivot(index='cen', columns='n', values='res')
lag1 = np.mean([rp[n].corr(rp[n + 1]) for n in range(1, 11)])

diag = pd.DataFrame({'teste': ['Breusch-Pagan (estatística LM)', 'Breusch-Pagan (p-valor)', 'Jarque-Bera (estatística)',
                               'Assimetria dos resíduos', 'Curtose em excesso', 'Pontos com Cook > 4/N',
                               '% de pontos com Cook > 4/N', 'Correlação dos resíduos entre anos seguidos (média)'],
                     'valor': [bp[0], bp[1], jb.statistic, stats.skew(res), stats.kurtosis(res),
                               int((cook > limite_cook).sum()), (cook > limite_cook).mean() * 100, lag1]})
diag.to_csv('tabelas/tabela_diagnosticos.csv', index=False)

rmse_ano = b11.assign(res=res).groupby('n')['res'].apply(lambda r: np.sqrt((r ** 2).mean())).rename('RMSE').reset_index()
rmse_ano.columns = ['Ano', 'RMSE']
rmse_ano.to_csv('tabelas/tabela_rmse_por_ano.csv', index=False)

print(vif.to_string(index=False))
print()
print(diag.to_string(index=False))
print()
print(rmse_ano.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))

ax[0, 0].scatter(m3_ols.fittedvalues, res, s=4, alpha=0.3, color=AZUL)
ax[0, 0].axhline(0, color=LARANJA, lw=1)
ax[0, 0].set_xlabel('Valores ajustados (R$ milhões)')
ax[0, 0].set_ylabel('Resíduos (R$ milhões)')
ax[0, 0].set_title('Resíduos x ajustados')

sm.qqplot(res, line='s', ax=ax[0, 1], markersize=2, markerfacecolor=AZUL, markeredgecolor=AZUL)
ax[0, 1].set_title('Gráfico Q-Q dos resíduos')

ax[1, 0].hist(res, bins=80, color=AZUL, edgecolor='white')
ax[1, 0].set_xlabel('Resíduo (R$ milhões)')
ax[1, 0].set_ylabel('Frequência')
ax[1, 0].set_title('Distribuição dos resíduos')

ax[1, 1].bar(rmse_ano['Ano'], rmse_ano['RMSE'], color=CINZA)
ax[1, 1].set_xticks(anos_ok)
ax[1, 1].set_xlabel('Ano')
ax[1, 1].set_ylabel('RMSE (R$ milhões)')
ax[1, 1].set_title('Erro do modelo em cada ano')

plt.tight_layout()
plt.savefig('figuras/fig05_diagnostico_residuos.png', dpi=150)
plt.show()

## 8. Validação em cenários novos
Separamos 70% dos **cenários** para treino e 30% para teste (todos os anos de um cenário ficam do mesmo lado, para não vazar informação). Comparamos o Modelo 3 com dois modelos mais simples.

In [ ]:
rng = np.random.default_rng(42)
cens = b11['cen'].unique()
cen_teste = rng.choice(cens, size=int(0.3 * len(cens)), replace=False)
teste = b11[b11['cen'].isin(cen_teste)]
treino = b11[~b11['cen'].isin(cen_teste)]

modelos = {'Só a média do ano': 'G ~ C(n)',
           'EBITDA + ano': 'G ~ EBITDA + C(n)',
           'EBITDA + CAPEX + DESPFIN + ano (Modelo 3)': formula3}
linhas = []
for nome, f in modelos.items():
    mt = smf.ols(f, treino).fit()
    pt = mt.predict(treino)
    pe = mt.predict(teste)
    linhas.append({'Modelo': nome,
                   'RMSE_treino': np.sqrt(np.mean((treino['G'] - pt) ** 2)),
                   'RMSE_teste': np.sqrt(np.mean((teste['G'] - pe) ** 2)),
                   'MAE_teste': mean_absolute_error(teste['G'], pe),
                   'R2_teste': r2_score(teste['G'], pe)})
t_val = pd.DataFrame(linhas)
t_val.to_csv('tabelas/tabela_validacao.csv', index=False)
print('Treino:', treino['cen'].nunique(), 'cenários | Teste:', teste['cen'].nunique(), 'cenários')
t_val

In [ ]:
mt = smf.ols(formula3, treino).fit()
pe = mt.predict(teste)
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(pe, teste['G'], s=4, alpha=0.35, color=AZUL)
lim = [min(pe.min(), teste['G'].min()), max(pe.max(), teste['G'].max())]
ax.plot(lim, lim, color=LARANJA, lw=1.5)
ax.set_xlabel('Geração de Caixa prevista (R$ milhões)')
ax.set_ylabel('Geração de Caixa observada (R$ milhões)')
ax.set_title('Cenários de teste: observado x previsto')
plt.tight_layout()
plt.savefig('figuras/fig06_observado_previsto_teste.png', dpi=150)
plt.show()

## 9. Conclusões e limitações
Os números finais e a interpretação completa estão no relatório Word. Resumo do que o notebook mostra:

- **O EBITDA sozinho explica pouco do caixa.** No Modelo 1 com os anos 1 a 11 juntos o R² é baixo, e dentro de cada ano ele varia de quase zero a cerca de 0,9. Isso apoia a tese de que lucro não vira caixa de forma proporcional.
- **CAPEX e despesas financeiras explicam a maior parte da diferença entre cenários** nos primeiros anos, o que eleva o R² do Modelo 2.
- **No Ano 12 a Distribuição para Acionista é o que importa.** Quando `DIST` entra no modelo, o EBITDA perde significância (p > 0,05).
- **Os pressupostos não são perfeitos:** resíduos com caudas pesadas, variância não constante e dependência entre anos do mesmo cenário. Por isso usamos erros-padrão robustos e agrupados.

**Limitações**
1. Os dados são simulados (Monte Carlo), então os resultados valem para o modelo da CTI e não para dados reais observados.
2. Correlação não é causalidade: o CAPEX, por exemplo, carrega o efeito de outros fatores do cenário.
3. A Geração de Caixa é uma soma de itens do fluxo, então parte do R² dos modelos com CAPEX e despesas financeiras é mecânica.
4. A `DIST` é fixa na maior parte dos anos, então não dá para estimar o efeito da política de dividendos nesses anos.
5. O Ano 12 tem contas ausentes (N/D) e liquidação da concessão, então não é comparável aos demais.